In [ ]:
import math
import numpy as np
import pandas as pd
from datetime import datetime
import time as time_mod

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from pyomo.environ import *
from pyomo.dae import *

from scipy.integrate import odeint, solve_ivp
from scipy.interpolate import CubicSpline
from scipy.optimize import least_squares
from sklearn.metrics import r2_score, mean_squared_error

from joblib import Parallel, delayed

import pickle
import random

import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import gc

# Generate Synthetic Raman Data

In [ ]:
def case_study1(t, y, Ki):
    Ca, Cb, Cc = y
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    v1max = 4.5
    v2max = 2.5
    
    Km1 = 5
    Km2 = 5
    
    Ki1 = Ki[0]
    Ki2 = Ki[1]
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    inh1 = (1 + (Cc/Ki1))
    inh2 = (1 + (Cc/Ki2))
    
#     # Uncompetitive Inhibition    
#     v1 = ((v1max/inh1)*Ca)/((Ca + Km1*inh1))
#     v2 = ((v2max/inh2)*Cb)/((Ca + Km2*inh2))
    
    # Noncompetitive Inhibition    
    v1 = (v1max*Ca)/((Ca + Km1)*inh1)
    v2 = (v2max*Cb)/((Cb + Km2)*inh2)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    dCadt = -v1
    dCbdt = v1 - v2
    dCcdt = v2
    
    return [dCadt, dCbdt, dCcdt]

# Iterative Raman Hybrid Methodology

In [ ]:
def layer_calculation(W, b, vector, layer_int, num_output_nodes, no_actf = 0):
    
    output = []
    
    for i in range(num_output_nodes):
        temp = b[layer_int][i, 0]
        for k in range(len(vector)):
            temp += W[layer_int][k, i]*vector[k]
        
        if no_actf == 0:
            h_act = 1 / (1 + np.exp(-1 * temp))
        else:
            h_act = temp
            
        h = h_act
        output.append(h)

    return output
    

def ANN_EzKin_model(input_vector, ANN_arch, W, b):
    
    for i in range(len(ANN_arch)-2):
        output_vector = layer_calculation(W, b, input_vector, i, ANN_arch[i+1])
        input_vector = output_vector

    j = i + 1
    output_vector = layer_calculation(W, b, input_vector, j, ANN_arch[j+1])
    
    fin = output_vector
    
    return fin

In [ ]:
def HMI_CS1_system(t, y, ANN_arch, params):
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    ca, cb, cc = y
    
    input_vector = [ca, cb, cc]

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    k1 = params[0]
    k2 = params[1]
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    W_len = 0
    b_len = 0
    for i in range(len(ANN_arch)-1):
        W_len += ANN_arch[i+1]*ANN_arch[i]
        b_len += ANN_arch[i+1]
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    W = []
    offset = 2
    for i in range(len(ANN_arch)-1):
        temp_arr = np.zeros(shape = (ANN_arch[i], ANN_arch[i+1]))
        
        r, c = np.shape(temp_arr)
        
        for j in range(r):
            for k in range(c):
                temp_arr[j, k] = params[offset]
                offset += 1
        
        W.append(temp_arr)
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    b = []
    offset = 2 + W_len
    for i in range(len(ANN_arch)-1):
        temp_arr = np.zeros(shape = (ANN_arch[i+1], 1))
        
        r, c = np.shape(temp_arr)
        
        for j in range(r):
            for k in range(c):
                temp_arr[j,k] = params[offset]
                offset += 1
                
        b.append(temp_arr)
        
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    output_vec =  ANN_EzKin_model(input_vector, ANN_arch, W, b)
    
    Ezkin1 = output_vec[0]
    Ezkin2 = output_vec[1]
    
    
    r1 = k1*Ezkin1*ca
    r2 = k2*Ezkin2*cb
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~    

    dcadt = -1*r1
    
    dcbdt = r1 - r2
    
    dccdt = r2
        
    return [dcadt, dcbdt, dccdt]

In [ ]:
def residuals(params, t_data, y_data, C0, ANN_arch, id_of_non):
    
    nd = len(C0)
    nt = len(t_data)
    
    y_pred_full = []
    
    for i in range(nd):
        
        y0 = C0[i]
        sol_pred = solve_ivp(HMI_CS1_system,
                     (t_data[0], t_data[-1]),
                     y0,
                     t_eval=t_data,
                     args=tuple([ANN_arch, params]),
                     method='Radau',
                     rtol=1e-6,
                     atol=1e-8)
        
        y_pred = sol_pred.y
        
        yp = []
        for k in range(len(y_pred)):
            if k not in id_of_non:
                yp.append(y_pred[k])
        
        yp = np.array(yp).T
        
        y_pred_full.append(yp)
    
    y_pred_full = np.vstack(y_pred_full)
    res = (y_pred_full - y_data).ravel()
    
    return res

In [ ]:
def eval_residuals(t_fin, C_subsection, num_params, ANN_arch, C0, id_of_non, lower_bound, upper_bound, rs_P0, wall_time=60):
    
    rng = np.random.default_rng(rs_P0)
    p0 = rng.standard_normal((num_params,))
    
    p0[0] = abs(p0[0])
    p0[1] = abs(p0[1])
    
    p0 = np.clip(p0, lower_bound, upper_bound)
    
    # --- wall time setup ---
    start_time = time_mod.time()
    
    def residuals_timed(params, t_data, y_data, C0, ANN_arch, id_of_non):
        if time_mod.time() - start_time > wall_time:
            raise TimeoutError("Wall time exceeded")
        return residuals(params, t_data, y_data, C0, ANN_arch, id_of_non)
    # -----------------------
    
    try:
        results = least_squares(residuals_timed,
                                x0=p0,
                                method="trf",
                                args=tuple([t_fin, C_subsection, C0, ANN_arch, id_of_non]),
                                bounds=(lower_bound, upper_bound))
        obj = results.cost
    except TimeoutError:
        obj = np.inf
    except Exception as e:
        obj = np.inf
    
    return obj

In [ ]:
def robust_least_squares(t_fin, C_subsection, num_params, ANN_arch, C0, id_of_non, lower_bound, upper_bound, n_attempts):

    best_result = None
    best_cost = np.inf
    
    ls_inp = []
    for seed in range(n_attempts):
        ls_inp.append(tuple([t_fin, C_subsection, num_params, ANN_arch, C0, id_of_non, lower_bound, upper_bound, seed]))
    
    objective_eval = Parallel(n_jobs = n_attempts)(
                    delayed(eval_residuals)(t, C, n, A, c0, ion, lb, uo, s) for t, C, n, A , c0, ion, lb, uo, s in ls_inp)
    
    objective_eval = np.array(objective_eval)
    
    n_errors = np.sum(np.isinf(objective_eval))
    if n_errors > 0:
        print(f"{n_errors}/{len(objective_eval)} initial conditions errored")
        
    best_seed = np.argmin(objective_eval)
    
    rng = np.random.default_rng(best_seed)
    p0 = rng.standard_normal((num_params,))
    
    p0[0] = abs(p0[0])
    p0[1] = abs(p0[1])
    
    p0 = np.clip(p0, lower_bound, upper_bound)
    best_result = least_squares(residuals, 
                                x0 = p0, 
                                method = "trf",
                                args = tuple([t_fin, C_subsection, C0, ANN_arch, id_of_non]),
                                bounds = (lower_bound, upper_bound)
                               )
    
    return best_result

In [ ]:
def simplisma(D, nc, noise_fraction=0.01):
    ntp, nwp = D.shape
    
    row_norms = np.linalg.norm(D, axis=1, keepdims=True)
    D_norm = D / (row_norms + 1e-10)   # shape (ntp x nwp)
    

    mu    = np.mean(D, axis=1) 
    sigma = np.std(D, axis=1)
    
    alpha = noise_fraction * np.mean(mu)

    purity = sigma / (mu + alpha)   
    
    purest_indices = []
    weights = np.ones(ntp)          
    
    for k in range(nc):
        
        weighted_purity = purity * weights
        
        best_idx = np.argmax(weighted_purity)
        purest_indices.append(best_idx)
        
        selected = D_norm[purest_indices, :]   
        
        for i in range(ntp):
            row = D_norm[i, :]                
            proj = selected.T @ np.linalg.lstsq(
                selected @ selected.T,
                selected @ row,
                rcond=None
            )[0]                              
            residual = row - proj
            weights[i] = np.linalg.norm(residual)
    
    S_init = D[np.sort(purest_indices), :].T         
    
    return S_init, np.sort(purest_indices)

In [ ]:
def mcr_als_hsm(D_full, C0, t_fin, ANN_arch, num_comp, nc, id_of_non, num_iter):
    
    S_init, purest_indices = simplisma(D_full, nc, 0.01)
    
    ST_iter = St_init
    LOF_list = []
    C_list = []
    ST_list = []
    P_list = []
    
    for n in range(num_iter):
        if (n+1)%10 == 0:
            print("Iteration " + str(n+1) + ": ", datetime.now())
            
        C_soft, residuals, rank, s = np.linalg.lstsq(ST_iter.T, D_full.T, rcond = None)
        C_soft = np.clip(C_soft, 0, None)
        
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        
        num_params = 2
        for i in range(len(ANN_arch)-1):
            num_params += ANN_arch[i+1]*ANN_arch[i]
            num_params += ANN_arch[i+1]

        C_subsection = C_soft[:nc,:].T

        lower_bound = np.full(num_params, -1.0, dtype = float)
        upper_bound = np.full(num_params, 1.0, dtype = float)
        
        for i in range(2):
            lower_bound[i] = 0
            upper_bound[i] = np.inf
            
        result = robust_least_squares(t_fin, C_subsection, num_params, ANN_arch, C0, 
                                      id_of_non, lower_bound, upper_bound, 25)
        
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        
        C_new_subsection = []
         
        nd = len(C0)
        pnew = result.x
        
        P_list.append(pnew)
        
        for i in range(nd):

            y0 = C0[i,:]
            sol_nw_pr = solve_ivp(HMI_CS1_system,
                             (t_fin[0], t_fin[-1]),
                             y0,
                             t_eval=t_fin,
                             args=tuple([ANN_arch, pnew]),
                             method='Radau',
                             rtol=1e-6,
                             atol=1e-8)
            C_nws = []
            for k in range(len(sol_nw_pr.y)):
                if k not in id_of_non:
                    C_nws.append(sol_nw_pr.y[k])
            
            C_nws = np.array(C_nws).T
            C_new_subsection.append(C_nws)

        C_new_subsection = np.vstack(C_new_subsection)
        
        C_hard = C_soft.T
        C_hard[:,:nc] = C_new_subsection
        
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        
        ST_iter_f, residuals, rank, s = np.linalg.lstsq(C_hard, D_full, rcond = None)
        ST_iter_f = np.clip(ST_iter_f, 0, None)
        ST_iter = ST_iter_f
        
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        
        D_hat = np.matmul(C_hard, ST_iter)
        D_err = D_full - D_hat
        LOF = np.linalg.norm(D_err)/np.linalg.norm(D_full)
        LOF_list.append(LOF)
        C_list.append(C_hard)
        ST_list.append(ST_iter)
        
    return [C_list, ST_list, P_list, np.array(LOF_list)]

In [ ]:
def predictive_perf_testing(ic, tspan, Ki, ANN_arch, params):
    
    r2_list = []
    rmse_list = []
    nrmse_list = []

    sol_true = solve_ivp(case_study1, 
                    tspan, 
                    ic,
                    args = Ki,
                    method = "Radau")
    
    teval = sol_true.t

    sol_calc = solve_ivp(HMI_CS1_system, 
                        tspan, 
                        ic, 
                        t_eval = teval, 
                        args = tuple([ANN_arch, params]), 
                        method = "Radau")

    y_true = sol_true.y
    y_calc = sol_calc.y

    for j in range(len(y_true)):
        mse = mean_squared_error(y_true[j], y_calc[j])
        rmse_list.append(mse**0.5)
        nrmse_list.append((mse**0.5)/np.mean(y_true[j]))
        
        if r2_score(y_true[j], y_calc[j]) < 0:
            r2_list.append(0)
        else:
            r2_list.append(r2_score(y_true[j], y_calc[j]))
        
    return tuple([np.mean(r2_list), np.mean(rmse_list), np.mean(nrmse_list)])

In [ ]:
def spectra_recovery_metrics(ST_gt, ST_final):
    
    groundtruth_spectra = ST_gt
    
    recovered_spectra = np.array(ST_final)
    
    spectral_metric = {}
    
    for i in range(len(groundtruth_spectra)):    
        gs = groundtruth_spectra[i,:]
        gs_n = gs / np.linalg.norm(gs)

        rs = recovered_spectra[i,:] 
        rs_n = rs / np.linalg.norm(rs)

        err = gs - rs

        sm = np.matmul(gs_n, rs_n.T)
        nrmse = np.linalg.norm(err)/np.linalg.norm(gs)
        
        ky = "Component " + str(i+1)
        spectral_metric[ky] = [sm, nrmse]
    
    return spectral_metric

In [ ]:
C1_range = np.logspace(-2, 1, 500)

rect_init = []

for i in range(len(C1_range)):
    rect_init.append(np.array([C1_range[i], 0, 0]))

In [ ]:
train_scenario3 = os.listdir("Train Conditions/Scenario 3")

In [ ]:
for i in range(len(train_scenario3)):
    
    file = os.path.join("Train Conditions/Scenario 3", train_scenario3[i])
    
    with open(file, "rb") as f:
        full_data = pickle.load(f)
        
    t_fin = full_data["t_fin"]
    wv_fin = full_data["wv_fin"].reshape(-1,)
    D_full = full_data["D_full"]
    C0 = full_data["C0"]
    ST_ground = full_data["ST_ground"]
    
    id_of_LD = [1]
    ANN_arch = [3, 2, 2]
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    print("\033[1mTime start of MCR-ALS: \033[0m", datetime.now())
    
    start_compute = time_mod.time()

    C_list, ST_list, P_list, LOF_list = mcr_als_hsm(D_full, C0, t_fin, ANN_arch, 2, 2, id_of_LD, 5)

    ideal_ind = np.argmin(LOF_list)
    C_final = C_list[ideal_ind]
    ST_final = ST_list[ideal_ind]
    P_final = P_list[ideal_ind]
    
    end_compute = time_mod.time()
    
    print("\033[1mTime end of MCR-ALS: \033[0m", datetime.now())

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    compute_time = end_compute - start_compute
    
    rs_ST = int(train_scenario3[i].split("_")[3])
    rs_D = float(train_scenario3[i].split("_")[-1].split(".")[0])

    Ki1 = float(train_scenario3[i].split("_")[1])
    Ki2 = float(train_scenario3[i].split("_")[2])
    Ki = tuple([[Ki1, Ki2]])
    spc_ovp = float(train_scenario3[i].split("_")[4])
    
    test2_input = []
    for m in range(len(rect_init)):
        test2_input.append(tuple([rect_init[m], (0, 20), Ki, ANN_arch, P_final]))

    conc_predictive_performance = Parallel(n_jobs = -1)(
        delayed(predictive_perf_testing)(ic, t, k, AA, PF) for ic, t, k, AA, PF in test2_input)

    spectra_predictive_performance = spectra_recovery_metrics(ST_ground, ST_final)
    
    final_performance_dict = {}
    final_performance_dict["conc_perf"] = conc_predictive_performance
    final_performance_dict["spectra_perf"] = spectra_predictive_performance
    final_performance_dict["Params"] = [P_final, ANN_arch]
    final_performance_dict["MCR Iterations"] = [C_list, ST_list, P_list, LOF_list]
    final_performance_dict["Compute Time"] = compute_time

    print("\033[1mTime end of Test: \033[0m", datetime.now())
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    path_name = "Iterative Performance/SIMPLISMA_init/Scenario 3/(322)_ANN/"
    file_name = "Perf_"+str(Ki[0][0])+"_"+str(Ki[0][1])+"_"+str(rs_ST)+"_"+str(spc_ovp)+"_"+str(rs_D)+".pkl"
    final_name = path_name + file_name

    with open(final_name, "wb") as f:
        pickle.dump(final_performance_dict, f)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")